<div align="center">
<a href="https://rapidfire.ai/"><img src="https://raw.githubusercontent.com/RapidFireAI/rapidfireai/main/docs/images/RapidFire - Blue bug -white text.svg" width="115"></a>
<a href="https://discord.gg/6vSTtncKNN"><img src="https://raw.githubusercontent.com/RapidFireAI/rapidfireai/main/docs/images/discord-button.svg" width="145"></a>
<a href="https://oss-docs.rapidfire.ai/"><img src="https://raw.githubusercontent.com/RapidFireAI/rapidfireai/main/docs/images/documentation-button.svg" width="125"></a>
<br/>
Join Discord if you need help + ⭐ <i>Star us on <a href="https://github.com/RapidFireAI/rapidfireai">GitHub</a></i> ⭐
<br/>
To install RapidFire AI on your own machine, see the <a href="https://oss-docs.rapidfire.ai/en/latest/walkthrough.html">Install and Get Started</a> guide in our docs.
</div>

### RapidFire AI RAG/Context Engineering Tutorial Use Case: SciFact Q&A Chatbot

In [1]:
# import getpass
# TRITON_API_KEY = getpass.getpass("Enter your Triton API key: ")

# The above option is probably safer
from pathlib import Path
TRITON_API_KEY = Path("~/Project1/api-key.txt").expanduser().read_text(encoding="utf-8").splitlines()[0].strip()

In [2]:
from rapidfireai.automl import (
    List,
    RFLangChainRagSpec,
    RFOpenAIAPIModelConfig,
    RFPromptManager,
    RFGridSearch,
)
###################### Import before Experiment
# from rapidfireai_datahub_compat import (
#     patch_rapidfireai_for_datahub
# )
# print(patch_rapidfireai_for_datahub())
######################
from rapidfireai import Experiment

import re, json
from typing import List as listtype, Dict, Any

INFO 04-26 23:06:00 [importing.py:44] Triton is installed but 0 active driver(s) found (expected 1). Disabling Triton to prevent runtime errors.
INFO 04-26 23:06:00 [importing.py:68] Triton not installed or not compatible; certain GPU-related functions will not be available.


##### ⚠️ API Cost Considerations
This notebook runs 4 configurations concurrently on a downsampled dataset of 256 examples.
Estimated Costs:
- Current run (downsampled): \$5 
- Full set: \$45

> 💡 **Tip:** Monitor your API usage to avoid unexpected charges.

### Load Queries to `qrels`

In [3]:
import pandas as pd
from datasets import Dataset
from pathlib import Path

# Load golden QA pairs from validation set
validation_file = Path("~/Project1/validation-set-golden-qa-pairs.json").expanduser()
with open(validation_file, "r") as f:
    qa_pairs = json.load(f)

# Build queries dataset
queries = {
    "query": [qa["question"] for qa in qa_pairs],
    "query_id": [qa["question_id"] for qa in qa_pairs],
    "reference_answer": [qa["reference_answer"] for qa in qa_pairs],
}
queries = Dataset.from_dict(queries).shuffle(seed=42)

# Build qrels from source_evidence
# qrels maps question_id to relevant documents (source files)
qrels_data = []
for qa in qa_pairs:
    query_id = qa["question_id"]

    for evidence in qa["source_evidence"]:
        file_name = evidence.get("file", "")
        lines = evidence.get("lines", [])
        qrels_data.append({
            "query_id": query_id,
            "corpus": file_name,  # Using filename as document ID
            "lines": lines,  
            "relevance": 1  # Mark as relevant
        })


qrels = pd.DataFrame(qrels_data)
print(f"Loaded {len(qa_pairs)} QA pairs")
print(f"Dataset shape: {queries.shape}")
print(f"Qrels shape: {qrels.shape}")
qrels.head(10)

Loaded 45 QA pairs
Dataset shape: (45, 3)
Qrels shape: (52, 4)


,query_id,corpus,lines,relevance
0,1,configs.rst,"[23, 48]",1
1,2,difference.rst,"[4, 29]",1
2,3,experiment.rst,"[27, 39]",1
3,4,configs.rst,"[58, 102]",1
4,5,walkthroughrag.rst,"[29, 103]",1
5,6,icops.rst,"[75, 176]",1
6,7,evalsfunctions.rst,"[8, 31]",1
7,8,ragspecs.rst,"[56, 98]",1
8,8,rag_fiqa.rst,"[49, 59]",1
9,9,experiment.rst,"[215, 219]",1


### Create Experiment

In [7]:
# from rapidfireai import Experiment
experiment = Experiment(experiment_name="exp1", mode="evals")

Experiment exp1 is currently running. Returning the same experiment object.
Any running tasks have been cancelled.


### Define Partial Multi-Config Knobs for LangChain part of RAG Pipeline using RapidFire AI Wrapper APIs

In [ ]:
from langchain_community.document_loaders import DirectoryLoader, JSONLoader
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_openai import OpenAIEmbeddings
from typing import Dict

batch_size = 32

def custom_template(retrieved_docs: listtype[Document]) -> str:
    """
    Custom prompt template that combines the content of retrieved documents into a single context string for the generator model.
    """
    formatted_docs = []
    for doc in retrieved_docs:
        content = doc.page_content
        formatted_doc = f"{content}"  # only include helpful content without metadata
        formatted_docs.append(formatted_doc)
    
    # Combine all formatted documents into a single context string
    context = "\n\n".join(formatted_docs)
    
    return context


# CPU-based RAG
rag_cpu = RFLangChainRagSpec(
    document_loader=DirectoryLoader(
        path="~/Project1/sourcedocs",
        glob="*.rst",
        loader_cls=JSONLoader,
        sample_seed=1337,
    ),
    text_splitter=List([
            RecursiveCharacterTextSplitter.from_tiktoken_encoder(
                encoding_name="gpt2", chunk_size=256, chunk_overlap=32
            ),
            RecursiveCharacterTextSplitter.from_tiktoken_encoder(
                encoding_name="gpt2", chunk_size=128, chunk_overlap=32
            ),
        ],
    ),
    embedding_cfg=List([
        # {
        #     "class": HuggingFaceEmbeddings,
        #     "model_name": "sentence-transformers/all-MiniLM-L6-v2",
        #     "model_kwargs": {"device": "cpu"},
        #     "encode_kwargs": {"normalize_embeddings": True, "batch_size": batch_size},
        # },
        { # For quicker demo in class
            "class": OpenAIEmbeddings,
            "model": "api-tgpt-embeddings",
            "api_key": TRITON_API_KEY,
            "base_url": "https://tritonai-api.ucsd.edu",
            "check_embedding_ctx_length": False,
        },
    ]),
    # FAISS is an in-memory store and only works in create mode. 
    vector_store_cfg={
        "type": "faiss", 
        "batch_size": batch_size
    }, # if not set, uses FAISS by default
    search_cfg=List([{"type": "similarity", "k": 10}, {"type": "mmr", "k": 10}]), # 2 different search types
    reranker_cfg={
        "class": CrossEncoderReranker,
        "model_name": "cross-encoder/ms-marco-MiniLM-L6-v2",
        "model_kwargs": {"device": "cpu"},
        "top_n": 5,
    },
    enable_gpu_search=False,
    document_template=custom_template,
)

### Define Data Processing and Postprocessing Functions

In [ ]:
def openai_sample_preprocess_fn(
    batch: Dict[str, listtype], rag: RFLangChainRagSpec, prompt_manager: RFPromptManager
) -> Dict[str, listtype]:
    """Function to prepare the final inputs given to the generator model with token limit"""
    import tiktoken

    SYSTEM_INSTRUCTIONS = "You are a helpful QA assistant. Answer the question based solely on the provided context. Be concise and accurate."
    MAX_TOKENS = 2000

    # Perform batched retrieval over all queries; returns a list of lists of k documents per query
    all_context = rag.get_context(batch_queries=batch["query"], serialize=False)

    # Extract the retrieved document ids from the context
    retrieved_documents = [
        [doc.meta["corpus"] for doc in docs] for docs in all_context
    ]

    # Serialize the retrieved documents into a single string per query using the default template
    serialized_context = rag.serialize_documents(all_context)
    batch["query_id"] = [int(query_id) for query_id in batch["query_id"]]

    # Token limiting function
    enc = tiktoken.get_encoding("gpt2")
    
    def truncate_prompt(prompt_text, max_tokens):
        """Truncate prompt to fit within token limit"""
        tokens = enc.encode(prompt_text)
        if len(tokens) > max_tokens:
            tokens = tokens[:max_tokens]
            return enc.decode(tokens)
        return prompt_text

    # Build prompts with token limit
    prompts = []
    for question, context in zip(batch["query"], serialized_context):
        prompt_content = f"Context:\n{context}\n\nQuestion: {question}\n\nAnswer:"
        prompt_content = truncate_prompt(prompt_content, MAX_TOKENS)
        
        prompts.append([
            {"role": "system", "content": SYSTEM_INSTRUCTIONS},
            {"role": "user", "content": prompt_content},
        ])

    # Each batch to contain conversational prompt, retrieved documents, and original 'query_id', 'query', 'metadata'
    return {
        "prompts": prompts,
        "retrieved_documents": retrieved_documents,
        **batch,
    }


def sample_postprocess_fn(batch: Dict[str, listtype]) -> Dict[str, listtype]:
    """Function to postprocess outputs produced by generator model"""
    import json
    from pathlib import Path
    
    # Load ground truth from validation set golden QA pairs
    golden_qa_file = Path("/home/yup031/Project1/validation-set-golden-qa-pairs.json")
    
    if golden_qa_file.exists():
        with open(golden_qa_file, 'r') as f:
            golden_qa_pairs = json.load(f)
        
        # Create a mapping from question to ground truth answer
        qa_map = {}
        for qa_pair in golden_qa_pairs:
            question_key = qa_pair.get("question") or qa_pair.get("query")
            answer_key = qa_pair.get("reference_answer") or qa_pair.get("ground_truth")
            if question_key:
                qa_map[question_key] = answer_key
        
        # Attach ground truth answers from validation set
        batch["ground_truth"] = [
            qa_map.get(query, "N/A")
            for query in batch.get("query", [])
        ]
    else:
        batch["ground_truth"] = ["N/A"] * len(batch.get("query", []))
    
    return batch

In [ ]:
def openai_sample_preprocess_fn(
    batch: Dict[str, listtype], rag: RFLangChainRagSpec, prompt_manager: RFPromptManager
) -> Dict[str, listtype]:
    """Function to prepare the final inputs given to the generator model"""

    all_context = rag.get_context(batch_queries=batch["query"], serialize=False)
    retrieved_documents = [
        [doc.metadata["corpus_id"] for doc in docs] for docs in all_context
    ]
    serialized_context = rag.serialize_documents(all_context)
    batch["query_id"] = [int(query_id) for query_id in batch["query_id"]]

    return {
        "prompts": [
            [
                {"role": "system", "content": INSTRUCTIONS},
                {
                    "role": "user",
                    "content": f"\nClaim:\n{question}. \nEvidence:\n{context}. \nYour response:",
                },
            ]
            for question, context in zip(batch["query"], serialized_context)
        ],
        "retrieved_documents": retrieved_documents,
        **batch,
    }


def extract_solution(answer):
    solution = re.search(r"####\s*(SUPPORT|CONTRADICT|NOINFO)", answer, re.IGNORECASE)
    if solution is None:
        return "INVALID"
    return solution.group(1).upper()


def sample_postprocess_fn(batch: Dict[str, listtype]) -> Dict[str, listtype]:
    """Function to postprocess outputs produced by generator model"""
    # Get ground truth documents for each query; can be done in preprocess_fn too but done here for clarity
    batch["ground_truth_documents"] = [
        qrels[qrels["query_id"] == query_id]["corpus_id"].tolist()
        for query_id in batch["query_id"]
    ]
    batch["answer"] = [extract_solution(answer) for answer in batch["generated_text"]]
    return batch

### Define Custom Eval Metrics Functions

In [ ]:
import math


def compute_ndcg_at_k(retrieved_docs: set, expected_docs: set, k=3):
    """Utility function to compute NDCG@k"""
    relevance = [1 if doc in expected_docs else 0 for doc in list(retrieved_docs)[:k]]
    dcg = sum(rel / math.log2(i + 2) for i, rel in enumerate(relevance))

    # IDCG: perfect ranking limited by min(k, len(expected_docs))
    ideal_length = min(k, len(expected_docs))
    ideal_relevance = [3] * ideal_length + [0] * (k - ideal_length)
    idcg = sum(rel / math.log2(i + 2) for i, rel in enumerate(ideal_relevance))

    return dcg / idcg if idcg > 0 else 0.0


def compute_rr(retrieved_docs: set, expected_docs: set):
    """Utility function to compute Reciprocal Rank (RR) for a single query"""
    rr = 0
    for i, retrieved_doc in enumerate(retrieved_docs):
        if retrieved_doc in expected_docs:
            rr = 1 / (i + 1)
            break
    return rr

def compute_accuracy(predictions, ground_truth):
    """Label prediction accuracy: SUPPORT, CONTRADICT, NOINFO"""
    return sum(1 for pred, gt in zip(predictions, ground_truth) if pred == gt) / len(predictions)

def sample_compute_metrics_fn(batch: Dict[str, listtype]) -> Dict[str, Dict[str, Any]]:
    """Function to compute all eval metrics based on retrievals and/or generations"""

    true_positives, precisions, recalls, f1_scores, ndcgs, rrs, acc = 0, [], [], [], [], [], []
    total_queries = len(batch["query"])

    for pred, gt in zip(batch["retrieved_documents"], batch["ground_truth_documents"]):
        expected_set = set(gt)
        retrieved_set = set(pred[:3])

        true_positives = len(expected_set.intersection(retrieved_set))
        precision = true_positives / len(retrieved_set) if len(retrieved_set) > 0 else 0
        recall = true_positives / len(expected_set) if len(expected_set) > 0 else 0
        f1 = (
            2 * precision * recall / (precision + recall)
            if (precision + recall) > 0
            else 0
        )

        precisions.append(precision)
        recalls.append(recall)
        f1_scores.append(f1)
        ndcgs.append(compute_ndcg_at_k(retrieved_set, expected_set, k=3))
        rrs.append(compute_rr(retrieved_set, expected_set))
    
    accuracy = compute_accuracy(batch["answer"], batch["label"])
        

    return {
        "Total": {"value": total_queries},
        "Precision": {"value": sum(precisions) / total_queries},
        "Recall": {"value": sum(recalls) / total_queries},
        "F1 Score": {"value": sum(f1_scores) / total_queries},
        "NDCG@3": {"value": sum(ndcgs) / total_queries},
        "MRR": {"value": sum(rrs) / total_queries},
        "Accuracy": {"value": accuracy}
    }


def sample_accumulate_metrics_fn(
    aggregated_metrics: Dict[str, listtype],
) -> Dict[str, Dict[str, Any]]:
    """Function to accumulate eval metrics across all batches"""

    num_queries_per_batch = [m["value"] for m in aggregated_metrics["Total"]]
    total_queries = sum(num_queries_per_batch)
    algebraic_metrics = ["Precision", "Recall", "F1 Score", "NDCG@3", "MRR", "Accuracy"]

    return {
        "Total": {"value": total_queries},
        **{
            metric: {
                "value": sum(
                    m["value"] * queries
                    for m, queries in zip(
                        aggregated_metrics[metric], num_queries_per_batch
                    )
                )
                / total_queries,
                "is_algebraic": True,
                "value_range": (0, 1),
            }
            for metric in algebraic_metrics
        },
    }

### Define Partial Multi-Config Knobs for OpenAI Generator part of RAG Pipeline using RapidFire AI Wrapper APIs

In [ ]:
# 2 generator configs with different sizes of generator models and different reasoning levels
openai_config = RFOpenAIAPIModelConfig(
    client_config={"api_key": TRITON_API_KEY, "base_url": "https://tritonai-api.ucsd.edu", "max_retries": 2},
    model_config={
        "model": "api-mistral-small-3.2-2506",
        "max_completion_tokens": 2048,
    },
    rpm_limit=120, # Request per minute (RPM) needs to be set based on your account tier and the specific model used
    tpm_limit=1_000_000, # Token per minute (TPM) needs to be set based on your account tier and the specific model used
    rag=rag_cpu,
    prompt_manager=None,
)


config_set = List([{
        "openai_config": openai_config,
        "batch_size": batch_size,
        "preprocess_fn": openai_sample_preprocess_fn,
        "postprocess_fn": sample_postprocess_fn,
        "compute_metrics_fn": sample_compute_metrics_fn,
        "accumulate_metrics_fn": sample_accumulate_metrics_fn,
        "online_strategy_kwargs": {
            "strategy_name": "normal",
            "confidence_level": 0.95,
            "use_fpc": True,
        },
    },
])

### Create Config Group

In [ ]:
# Simple grid search across all sets of config knob values = 4 combinations in total
config_group = RFGridSearch(config_set)

### Run Multi-Config Evals

In [ ]:
# Launch evals of all RAG configs in the config_group with swap granularity of 4 chunks
results = experiment.run_evals(
    config_group=config_group,
    dataset=scifact_dataset,
    num_shards=4,
    seed=42,
)

### View Results

In [ ]:
# Convert results dict to DataFrame
results_df = pd.DataFrame([
    {k: v['value'] if isinstance(v, dict) and 'value' in v else v for k, v in {**metrics_dict, 'run_id': run_id}.items()}
    for run_id, (_, metrics_dict) in results.items()
])

results_df

### End Experiment

In [ ]:
experiment.end()

### View RapidFire AI Log Files

In [ ]:
# Get the experiment-specific log file
log_file = experiment.get_log_file_path()

print(f"📄 Log File: {log_file}")
print()

if log_file.exists():
    print("=" * 80)
    print(f"Last 30 lines of {log_file.name}:")
    print("=" * 80)
    with open(log_file, 'r', encoding='utf-8') as f:
        lines = f.readlines()
        for line in lines[-30:]:
            print(line.rstrip())
else:
    print(f"❌ Log file not found: {log_file}")